In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd



# 1. map taxi zones to nearest weather station
def build_zone_station_map(shapefile_path):
    """Create a lookup table assigning each pickup zone to the nearest weather station."""
    zones = gpd.read_file(shapefile_path)
    zones = zones.rename(columns={"OBJECTID": "PULocationID"})
    zones["PULocationID"] = zones["PULocationID"].astype(int)

    # mompute centroids in projected CRS for accurate distance
    zones = zones.to_crs(epsg=2263)
    zones["centroid"] = zones.geometry.centroid

    # convert centroid coordinates back to lat/lon (EPSG:4326)
    centroids = gpd.GeoSeries(zones["centroid"], crs="EPSG:2263").to_crs(epsg=4326)
    zones["lon"], zones["lat"] = centroids.x, centroids.y

    # ref points for weather stations
    stations = pd.DataFrame({
        "station": ["laguardia", "jfk", "central_park"],
        "lat": [40.7792, 40.6398, 40.7812],
        "lon": [-73.88, -73.7789, -73.9665]
    })

    # find nearest station for each zone
    def nearest_station(lat, lon):
        dists = (stations["lat"] - lat)**2 + (stations["lon"] - lon)**2
        return stations.loc[dists.idxmin(), "station"]

    zones["station"] = zones.apply(lambda r: nearest_station(r["lat"], r["lon"]), axis=1)
    return zones[["PULocationID", "station"]].astype({"PULocationID": int})



# 2. clean and prep single weather station CSV

def clean_weather_file(filepath, station_name):
    """Load hourly weather CSV and standardize columns."""
    df = pd.read_csv(filepath)
    df["pickup_hour"] = pd.to_datetime(df["DATE"], errors="coerce").dt.floor("h")

    keep = [
        "pickup_hour",
        "HourlyPrecipitation",
        "HourlyDryBulbTemperature",
        "HourlyWindSpeed",
        "HourlyVisibility",
    ]
    df = df[[c for c in keep if c in df.columns]].copy()

    df = df.rename(columns={
        "HourlyPrecipitation": "precipitation",
        "HourlyDryBulbTemperature": "temperature",
        "HourlyWindSpeed": "wind_speed",
        "HourlyVisibility": "visibility",
    })

    for col in ["precipitation", "temperature", "wind_speed", "visibility"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = (
        df.groupby("pickup_hour", as_index=False)
          .mean(numeric_only=True)
          .sort_values("pickup_hour")
          .reset_index(drop=True)
    )
    df["station"] = station_name
    return df


# 3. combine weather data from all stations and fill missing hours

def build_weather_year(year, laguardia_path, central_park_path, jfk_path):
    """Combine LGA, JFK, and Central Park weather data for the year and fill gaps."""
    lga = clean_weather_file(laguardia_path, "laguardia")
    cp  = clean_weather_file(central_park_path, "central_park")
    jfk = clean_weather_file(jfk_path, "jfk")

    weather = pd.concat([lga, cp, jfk], ignore_index=True)
    weather["pickup_hour"] = pd.to_datetime(weather["pickup_hour"])

    # build full hourly grid for all stations
    full_hours = pd.date_range(f"{year}-01-01", f"{year}-12-31 23:00:00", freq="h")
    stations = sorted(weather["station"].unique())
    full_index = pd.MultiIndex.from_product([full_hours, stations],
                                            names=["pickup_hour", "station"])
    weather_full = (
        weather.set_index(["pickup_hour", "station"])
               .reindex(full_index)
               .reset_index()
               .sort_values(["station", "pickup_hour"])
    )

    # interpolate missing numeric values over time
    cols = ["precipitation", "temperature", "wind_speed", "visibility"]
    filled = []
    for st, grp in weather_full.groupby("station", group_keys=False):
        grp = grp.set_index("pickup_hour")
        for col in cols:
            grp[col] = grp[col].interpolate(method="time", limit_direction="both")
        filled.append(grp.reset_index())
    return pd.concat(filled, ignore_index=True)



# 4. merge taxi demand with weather data and create lag features

def build_taxi_weather_year(taxi_df, weather_df, zone_station_map):
    """Attach matching hourly weather info and lag features to taxi demand."""
    taxi = taxi_df.copy()
    taxi["PULocationID"] = taxi["PULocationID"].astype(int)
    taxi["pickup_hour"] = pd.to_datetime(taxi["pickup_hour"]).dt.floor("h")

    # merge with zone-to-station map
    taxi = taxi.drop(columns=["station"], errors="ignore").merge(
        zone_station_map, on="PULocationID", how="left"
    )

    # merge with hourly weather
    merged = taxi.merge(weather_df, on=["pickup_hour", "station"], how="left")

    # geenerate simple rain intensity feature
    merged["rain_category"] = np.select(
        [
            merged["precipitation"] == 0,
            (merged["precipitation"] > 0) & (merged["precipitation"] <= 0.5),
            merged["precipitation"] > 0.5,
        ],
        [0, 1, 2],
        default=0,
    )

    # Sort so groupby-shift works correctly
    merged = merged.sort_values(["PULocationID", "pickup_hour"])

    # Create lagged demand features
    for suffix, offset in [("lag_1", 1), ("lag_2", 2), ("lag_24", 24), ("lag_168", 168)]:
        merged[suffix] = merged.groupby("PULocationID")["trip_count"].shift(offset)

    return merged.reset_index(drop=True)



# 5. load pre-cleaned hourly taxi demand parquet from prev script

def load_taxi_year(year):
    taxi = pd.read_parquet(f"yellow_hourly_demand_full_{year}.parquet")
    taxi["PULocationID"] = taxi["PULocationID"].astype(int)
    taxi["pickup_hour"] = pd.to_datetime(taxi["pickup_hour"]).dt.floor("h")
    return taxi



# 6. run process for all years
year_configs = {
    2022: {
        "laguardia": "LaGuardia/LCD_USW00014732_2022_HourlyPrecipitation_nonempty.csv",
        "central_park": "Central_park/LCD_USW00094728_2022_HourlyPrecipitation_nonempty.csv",
        "jfk": "JFK_airport/LCD_USW00094789_2022_HourlyPrecipitation_nonempty.csv",
    },
    2023: {
        "laguardia": "LaGuardia/LCD_USW00014732_2023_HourlyPrecipitation_nonempty.csv",
        "central_park": "Central_park/LCD_USW00094728_2023_HourlyPrecipitation_nonempty.csv",
        "jfk": "JFK_airport/LCD_USW00094789_2023_HourlyPrecipitation_nonempty.csv",
    },
    2024: {
        "laguardia": "LaGuardia/LCD_USW00014732_2024_HourlyPrecipitation_nonempty.csv",
        "central_park": "Central_park/LCD_USW00094728_2024_HourlyPrecipitation_nonempty.csv",
        "jfk": "JFK_airport/LCD_USW00094789_2024_HourlyPrecipitation_nonempty.csv",
    },
    2025: {
        "laguardia": "LaGuardia/LCD_USW00014732_2025_HourlyPrecipitation_nonempty.csv",
        "central_park": "Central_park/LCD_USW00094728_2025_HourlyPrecipitation_nonempty.csv",
        "jfk": "JFK_airport/LCD_USW00094789_2025_HourlyPrecipitation_nonempty.csv",
    },
}


def main():
    zone_station_map = build_zone_station_map("taxi_zones.shp")
    results = {}

    for year, cfg in year_configs.items():
        print(f"\nProcessing {year}...")

        taxi = load_taxi_year(year)
        weather = build_weather_year(year, **cfg)
        taxi_weather = build_taxi_weather_year(taxi, weather, zone_station_map)

        taxi_weather.to_parquet(f"taxi_weather_{year}.parquet", index=False)
        results[year] = taxi_weather

        print(f"  Saved taxi_weather_{year}.parquet ({len(taxi_weather):,} rows)")
        print(f"  Missing temperature values: {taxi_weather['temperature'].isna().sum():,}")

    # quick check on one file
    df = pd.read_parquet("taxi_weather_2023.parquet")
    print("\nSample columns:", df.columns.tolist())
    print(df.head())

    return results


if __name__ == "__main__":
    main()
